# Explainability-Driven Analysis of Transformer Ensemble Calibration Failure in Indonesian Social Media Sentiment Analysis

---

## Overview

This notebook implements the **full experimental pipeline** for the paper:

> *"Explainability-Driven Analysis of Transformer Ensemble Calibration Failure in Indonesian Social Media Sentiment Analysis"*  
> 2026 International Conference on Information Management and Technology (ICIMTech), IEEE, pp. 96–101.  
> DOI: [10.1109/ICIMTech202671078.2026.11698984](https://doi.org/10.1109/ICIMTech202671078.2026.11698984) · [IEEE Xplore](https://ieeexplore.ieee.org/document/11698984)

### Research Goal
Build an **explainable stacking ensemble** for 3-class sentiment classification (*positive*, *neutral*, *negative*) using:

| Component | Role |
|---|---|
| **IndoBERT** (`indobert-base-p1`) | Transformer base model |
| **IndoBERTweet** (`indobertweet-base-uncased`) | Social-media-adapted transformer |
| **TF-IDF + LinearSVC** | Classical NLP baseline |
| **Logistic Regression** | Stacking meta-learner |

### Dataset
- **IndoNLU SmSA** — official train / val / test splits
- 3 classes: `positive` · `neutral` · `negative`

### Checkpoint & Resume Support
This notebook is **fully resumable**. Every trained model, probability array, and output file is persisted to disk. On subsequent runs, already-completed steps are **skipped automatically** — only missing checkpoints are re-computed.

```
outputs/
├── models/
│   ├── indobert/          ← HuggingFace fine-tuned model
│   ├── indobertweet/      ← HuggingFace fine-tuned model
│   ├── tfidf.pkl          ← TF-IDF vectorizer
│   ├── svm_clf.pkl        ← Calibrated SVM
│   ├── meta_learner.pkl   ← LR meta-learner
│   ├── timing.json        ← Training times
│   └── probas/            ← Probability arrays (.npy)
├── metrics/               ← CSV tables for paper
├── figures/               ← PNG figures (300 DPI)
└── predictions/           ← Prediction & failure CSVs
```

> **Runtime:** Google Colab with GPU (T4 or higher)  
> **First run:** ~90–120 min | **Subsequent runs:** ~5 min

## Section 0 — Installation

Run this cell once per Colab session to install all required libraries.

In [4]:
# Version bounds match requirements.txt:
#   datasets < 4.0      -> still supports the script-based `indonlp/indonlu` loader
#   scikit-learn < 1.7  -> still accepts LogisticRegression(multi_class='multinomial')
!pip install -q torch "transformers>=4.35" "datasets>=2.14,<4.0" "scikit-learn>=1.3,<1.7" shap matplotlib numpy pandas tqdm accelerate

## Section 0b — Working Directory (Google Colab or Local)

Every path in this notebook is relative to the **repository root** (e.g. `outputs/metrics/...`). This cell detects the environment automatically:

- **Google Colab** — mounts Google Drive so checkpoints **persist between sessions** (everything in `/content/` is lost when the runtime disconnects). Change `BASE_DIR` to use a different Drive folder.
- **Local Jupyter** — Jupyter starts the kernel inside `notebooks/`, so the cell moves one level up to the repository root.

In [5]:
# ── Working directory setup ──────────────────────────────────────────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Persist checkpoints on Google Drive between Colab sessions
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Explainable-Transformer'
    os.makedirs(BASE_DIR, exist_ok=True)
    os.chdir(BASE_DIR)
elif os.path.basename(os.getcwd()) == 'notebooks':
    # Local Jupyter: move from notebooks/ to the repository root
    os.chdir('..')

print(f'Working directory: {os.getcwd()}')

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Explainable-Transformer


## Section 1 — Imports, Configuration & Seeds

Global seed **42** is applied across Python / NumPy / PyTorch for reproducibility.

### Key Hyperparameters
| Parameter | Value | Rationale |
|---|---|---|
| `MAX_LEN` | 128 | Sufficient for short social media texts |
| `BATCH_SIZE` | 16 | Fits T4 GPU memory |
| `EPOCHS` | 3 | Standard BERT fine-tuning |
| `LR` | 2e-5 | BERT authors' recommendation |
| `WARMUP_RATIO` | 0.1 | 10 % linear warmup |

### `FORCE_RETRAIN` Flag
Set `FORCE_RETRAIN = True` to **ignore all saved checkpoints** and retrain from scratch. Leave it `False` to resume from where you left off.

In [6]:
import os, sys, time, warnings, random, re, json, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
import shap
from tqdm.auto import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'✓ GPU    : {torch.cuda.get_device_name(0)}')

# ── Checkpoint control ────────────────────────────────────────────────────────
FORCE_RETRAIN = False   # Set True to retrain everything from scratch

# ── Model Checkpoints ─────────────────────────────────────────────────────────
INDOBERT_CKPT     = 'indobenchmark/indobert-base-p1'
INDOBERTWEET_CKPT = 'indolem/indobertweet-base-uncased'
# Fallback: 'cahya/bert-base-indonesian-522M'

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LEN      = 128
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1

LABEL2ID   = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3

# ── Persistent paths ──────────────────────────────────────────────────────────
PATHS = {
    'ib_dir'    : 'outputs/models/indobert',
    'ibt_dir'   : 'outputs/models/indobertweet',
    'tfidf'     : 'outputs/models/tfidf.pkl',
    'svm'       : 'outputs/models/svm_clf.pkl',
    'meta'      : 'outputs/models/meta_learner.pkl',
    'timing'    : 'outputs/models/timing.json',
    'probas_dir': 'outputs/models/probas',
}

for d in ['outputs/metrics', 'outputs/figures', 'outputs/predictions',
           PATHS['ib_dir'], PATHS['ibt_dir'], PATHS['probas_dir']]:
    os.makedirs(d, exist_ok=True)

print('✓ Directories ready.')
print(f'✓ FORCE_RETRAIN = {FORCE_RETRAIN}')

✓ Device : cuda
✓ GPU    : Tesla T4
✓ Directories ready.
✓ FORCE_RETRAIN = False


## Section 2 — Checkpoint Utilities

These helper functions power the **save / load / skip** logic throughout the notebook.

| Function | Purpose |
|---|---|
| `ckpt_exists(key)` | Returns `True` if a checkpoint is already saved |
| `save_pickle / load_pickle` | Persist sklearn models (TF-IDF, SVM, LR) |
| `save_npy / load_npy` | Persist NumPy probability arrays |
| `save_timing / load_timing` | Persist training wall-clock times as JSON |
| `status(msg, loaded)` | Pretty-print whether something was loaded or freshly trained |

In [7]:
def ckpt_exists(key):
    """Check whether a checkpoint file/directory already exists."""
    path = PATHS[key]
    if key in ('ib_dir', 'ibt_dir'):
        return os.path.isfile(os.path.join(path, 'config.json'))
    return os.path.isfile(path)

def probas_exist(name):
    """Check whether all six proba arrays for a model exist."""
    for split in ('train', 'val', 'test'):
        if not os.path.isfile(f"{PATHS['probas_dir']}/{name}_{split}.npy"):
            return False
    return True

def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_npy(arr, name, split):
    np.save(f"{PATHS['probas_dir']}/{name}_{split}.npy", arr)

def load_npy(name, split):
    return np.load(f"{PATHS['probas_dir']}/{name}_{split}.npy")

def save_timing(key, seconds):
    timing = {}
    if os.path.isfile(PATHS['timing']):
        with open(PATHS['timing']) as f:
            timing = json.load(f)
    timing[key] = seconds
    with open(PATHS['timing'], 'w') as f:
        json.dump(timing, f, indent=2)

def load_timing(key, default=0.0):
    if not os.path.isfile(PATHS['timing']):
        return default
    with open(PATHS['timing']) as f:
        return json.load(f).get(key, default)

def status(msg, loaded=True):
    icon = '📂 LOADED' if loaded else '🏋️ TRAINED'
    print(f'  {icon} — {msg}')

print('✓ Checkpoint utilities defined.')

✓ Checkpoint utilities defined.


## Section 3 — Load Dataset

We use the **IndoNLU SmSA** dataset (`indonlp/indonlu`, config `smsa`).  
Labels: `negative=0`, `neutral=1`, `positive=2`.

> The `label` field can be an `int` or `str` depending on the dataset version — both are handled.

In [8]:
from datasets import load_dataset

LABEL2ID = {
    "positive": 0,
    "neutral": 1,
    "negative": 2,
    "pos": 0,
    "neu": 1,
    "neg": 2,
}

print("Loading IndoNLU SmSA dataset...")

dataset = load_dataset("indonlp/indonlu", "smsa", trust_remote_code=True)

print(dataset)
print("\nColumns:", dataset["train"].column_names)
print("\nSample:", dataset["train"][0])


def get_texts_labels(split):
    texts, labels = [], []

    for ex in dataset[split]:
        text = ex.get("text") or ex.get("sentence") or ex.get("tweet")
        label = ex.get("label")

        texts.append(text)

        if isinstance(label, int):
            labels.append(label)
        else:
            labels.append(LABEL2ID[str(label).lower()])

    return texts, labels


train_texts, train_labels = get_texts_labels("train")
val_texts, val_labels = get_texts_labels("validation")
test_texts, test_labels = get_texts_labels("test")

print(f"\n✓ Train : {len(train_texts):,} samples")
print(f"✓ Val   : {len(val_texts):,} samples")
print(f"✓ Test  : {len(test_texts):,} samples")

Loading IndoNLU SmSA dataset...


README.md: 0.00B [00:00, ?B/s]

indonlu.py: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

The repository for indonlp/indonlu contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/indonlp/indonlu.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/11000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1260
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 500
    })
})

Columns: ['text', 'label']

Sample: {'text': 'warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !', 'label': 0}

✓ Train : 11,000 samples
✓ Val   : 1,260 samples
✓ Test  : 500 samples


## Section 4 — Exploratory Data Analysis

Inspect the **class distribution** across splits to check for label imbalance.

### Figure 1 → `fig1_class_distribution.png`
Insert in **Section 4 (Dataset)** of your paper. The figure is regenerated every run (fast — no model needed).

In [9]:
def class_counts(labels):
    return {ID2LABEL[i]: labels.count(i) for i in range(NUM_LABELS)}

train_counts = class_counts(train_labels)
val_counts   = class_counts(val_labels)
test_counts  = class_counts(test_labels)

stats_df = pd.DataFrame({'Train': train_counts, 'Validation': val_counts, 'Test': test_counts})
print(stats_df)
stats_df.to_csv('outputs/metrics/dataset_stats.csv')

colors = ['#d62728', '#aec7e8', '#2ca02c']
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (sname, counts) in zip(axes, [('Train', train_counts), ('Validation', val_counts), ('Test', test_counts)]):
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='black')
    ax.set_title(f'{sname} Set', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count'); ax.set_xlabel('Sentiment')
    for bar, v in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=11)
plt.suptitle('Class Distribution Across Splits', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig1_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig1_class_distribution.png | dataset_stats.csv')

          Train  Validation  Test
negative   6416         735   208
neutral    1148         131    88
positive   3436         394   204
✓ Saved fig1_class_distribution.png | dataset_stats.csv


## Section 5 — Text Preprocessing

| Pipeline | Applied To | Steps |
|---|---|---|
| `preprocess_raw` | IndoBERT, IndoBERTweet | Remove URLs · mentions · normalize hashtags · lowercase |
| `preprocess_svm` | TF-IDF + SVM | `preprocess_raw` **+** slang normalization |

> BERT models receive raw cleaned text — slang normalization would corrupt subword tokenization.

In [10]:
SLANG_DICT = {
    'gk':'tidak', 'ga':'tidak', 'gak':'tidak', 'tdk':'tidak',
    'udh':'sudah', 'udah':'sudah', 'sdh':'sudah',
    'aja':'saja', 'tp':'tapi', 'dgn':'dengan', 'dg':'dengan',
    'krn':'karena', 'krna':'karena', 'karna':'karena',
    'lg':'lagi', 'jg':'juga', 'sy':'saya', 'gue':'saya', 'gw':'saya',
    'lo':'kamu', 'lu':'kamu', 'km':'kamu',
    'hrs':'harus', 'msh':'masih', 'blm':'belum',
    'brp':'berapa', 'bgt':'banget', 'dr':'dari', 'utk':'untuk',
    'yg':'yang', 'org':'orang', 'bs':'bisa', 'bsa':'bisa',
    'jd':'jadi', 'sdg':'sedang', 'emg':'memang',
}

def basic_clean(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def preprocess_raw(text): return basic_clean(text)
def preprocess_svm(text):
    return ' '.join(SLANG_DICT.get(t, t) for t in basic_clean(text).split())

train_clean = [preprocess_raw(t) for t in train_texts]
val_clean   = [preprocess_raw(t) for t in val_texts]
test_clean  = [preprocess_raw(t) for t in test_texts]
train_svm   = [preprocess_svm(t) for t in train_texts]
val_svm     = [preprocess_svm(t) for t in val_texts]
test_svm    = [preprocess_svm(t) for t in test_texts]

print(f'Original : {train_texts[0]}')
print(f'Cleaned  : {train_clean[0]}')
print(f'SVM ver. : {train_svm[0]}')
print('\n✓ Preprocessing complete.')

Original : warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !
Cleaned  : warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !
SVM ver. : warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak ,

## Section 6 — PyTorch Dataset & Training Utilities

Reusable components shared by both transformer models:

- **`SentimentDataset`** — tokenises and wraps text + labels for `DataLoader`
- **`predict_logits()`** — batched GPU inference returning raw logits
- **`train_transformer()`** — AdamW + linear warmup scheduler + best-checkpoint saving
- **`get_probas_timed()`** — softmax probabilities + wall-clock inference time

In [11]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels':         torch.tensor(self.labels[idx], dtype=torch.long)}

def predict_logits(model, dataloader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            out = model(input_ids=batch['input_ids'].to(DEVICE),
                        attention_mask=batch['attention_mask'].to(DEVICE))
            all_logits.append(out.logits.cpu().numpy())
            all_labels.extend(batch['labels'].numpy())
    return np.vstack(all_logits), np.array(all_labels)

def train_transformer(model_name, ckpt, tr_texts, tr_labels, vl_texts, vl_labels):
    print(f'  ► Loading base model: {ckpt}')
    tok   = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(
        ckpt, num_labels=NUM_LABELS, ignore_mismatched_sizes=True).to(DEVICE)
    tr_dl = DataLoader(SentimentDataset(tr_texts, tr_labels, tok), BATCH_SIZE, shuffle=True)
    vl_dl = DataLoader(SentimentDataset(vl_texts, vl_labels, tok), BATCH_SIZE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(tr_dl) * EPOCHS
    sch   = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total), total)
    best_f1 = -1; t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); total_loss = 0
        for batch in tqdm(tr_dl, desc=f'    Epoch {ep}/{EPOCHS}'):
            out = model(input_ids=batch['input_ids'].to(DEVICE),
                        attention_mask=batch['attention_mask'].to(DEVICE),
                        labels=batch['labels'].to(DEVICE))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
            total_loss += out.loss.item()
        logits, true = predict_logits(model, vl_dl)
        preds = np.argmax(logits, 1)
        vf1  = f1_score(true, preds, average='macro', zero_division=0)
        vacc = accuracy_score(true, preds)
        print(f'    Ep {ep}: loss={total_loss/len(tr_dl):.4f}  val_acc={vacc:.4f}  val_f1={vf1:.4f}')
        if vf1 > best_f1:
            best_f1 = vf1
            model.save_pretrained(f'outputs/models/{model_name}')
            tok.save_pretrained(f'outputs/models/{model_name}')
    elapsed = time.time() - t0
    print(f'  ✓ Best val F1={best_f1:.4f}  |  Training time={elapsed:.1f}s')
    return elapsed

def get_probas_timed(model, tok, texts, labels):
    ds = SentimentDataset(texts, labels, tok)
    dl = DataLoader(ds, BATCH_SIZE)
    t0 = time.time()
    logits, true = predict_logits(model, dl)
    inf_t = time.time() - t0
    return torch.softmax(torch.tensor(logits), dim=-1).numpy(), true, inf_t

def gpu_mb():
    return torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0

print('✓ Training utilities defined.')

✓ Training utilities defined.


## Section 7 — Fine-tune IndoBERT *(with checkpoint resume)*

[**IndoBERT**](https://huggingface.co/indobenchmark/indobert-base-p1) is pre-trained on ~4 GB of Indonesian text.

**Checkpoint logic:**
- If `outputs/models/indobert/config.json` exists **and** `FORCE_RETRAIN=False` → load from disk, skip training
- Otherwise → fine-tune for `EPOCHS` epochs and save the best checkpoint

> First-run training on T4 GPU: ~20–30 minutes

In [12]:
if not FORCE_RETRAIN and ckpt_exists('ib_dir'):
    status('IndoBERT checkpoint found — loading from disk', loaded=True)
    indobert_model = AutoModelForSequenceClassification.from_pretrained(PATHS['ib_dir']).to(DEVICE)
    indobert_tok   = AutoTokenizer.from_pretrained(PATHS['ib_dir'])
    ib_train_time  = load_timing('indobert')
else:
    print('Fine-tuning IndoBERT...')
    ib_train_time = train_transformer(
        'indobert', INDOBERT_CKPT,
        train_clean, train_labels, val_clean, val_labels
    )
    save_timing('indobert', ib_train_time)
    indobert_model = AutoModelForSequenceClassification.from_pretrained(PATHS['ib_dir']).to(DEVICE)
    indobert_tok   = AutoTokenizer.from_pretrained(PATHS['ib_dir'])
    status('IndoBERT trained and saved', loaded=False)

print(f'  Params : {sum(p.numel() for p in indobert_model.parameters())/1e6:.1f}M')
print(f'  Timing : {ib_train_time:.1f}s')

Fine-tuning IndoBERT...
  ► Loading base model: indobenchmark/indobert-base-p1


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best val F1=0.9270  |  Training time=833.8s


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  🏋️ TRAINED — IndoBERT trained and saved
  Params : 124.4M
  Timing : 833.8s


## Section 8 — Fine-tune IndoBERTweet *(with checkpoint resume)*

[**IndoBERTweet**](https://huggingface.co/indolem/indobertweet-base-uncased) is pre-trained specifically on Indonesian Twitter data — ideal for social media sentiment.

> **If the checkpoint fails to load**, change `INDOBERTWEET_CKPT` to `'cahya/bert-base-indonesian-522M'`

Same fine-tuning strategy as IndoBERT for fair comparison.

In [13]:
if not FORCE_RETRAIN and ckpt_exists('ibt_dir'):
    status('IndoBERTweet checkpoint found — loading from disk', loaded=True)
    indobertweet_model = AutoModelForSequenceClassification.from_pretrained(PATHS['ibt_dir']).to(DEVICE)
    indobertweet_tok   = AutoTokenizer.from_pretrained(PATHS['ibt_dir'])
    ibt_train_time     = load_timing('indobertweet')
else:
    print('Fine-tuning IndoBERTweet...')
    ibt_train_time = train_transformer(
        'indobertweet', INDOBERTWEET_CKPT,
        train_clean, train_labels, val_clean, val_labels
    )
    save_timing('indobertweet', ibt_train_time)
    indobertweet_model = AutoModelForSequenceClassification.from_pretrained(PATHS['ibt_dir']).to(DEVICE)
    indobertweet_tok   = AutoTokenizer.from_pretrained(PATHS['ibt_dir'])
    status('IndoBERTweet trained and saved', loaded=False)

print(f'  Params : {sum(p.numel() for p in indobertweet_model.parameters())/1e6:.1f}M')
print(f'  Timing : {ibt_train_time:.1f}s')

Fine-tuning IndoBERTweet...
  ► Loading base model: indolem/indobertweet-base-uncased


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

... (9 outputs left out)

    Epoch 3/3:   0%|          | 0/688 [00:00<?, ?it/s]

    Ep 3: loss=0.0710  val_acc=0.9492  val_f1=0.9300
  ✓ Best val F1=0.9319  |  Training time=823.3s


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  🏋️ TRAINED — IndoBERTweet trained and saved
  Params : 110.6M
  Timing : 823.3s


## Section 9 — TF-IDF + LinearSVC *(with checkpoint resume)*

| Component | Setting |
|---|---|
| `TfidfVectorizer` | unigrams + bigrams · 50 K features · sublinear TF |
| `LinearSVC` | C=1.0 · max_iter=5 000 |
| `CalibratedClassifierCV` | 3-fold probability calibration |

Both the **TF-IDF vectorizer** and the **calibrated SVM** are saved as `pkl` files and reloaded on subsequent runs.

> Probability calibration is mandatory — the stacking ensemble requires well-calibrated `[P(neg), P(neu), P(pos)]` vectors from every base model.

In [14]:
if not FORCE_RETRAIN and ckpt_exists('tfidf') and ckpt_exists('svm'):
    status('TF-IDF + SVM checkpoint found — loading from disk', loaded=True)
    tfidf   = load_pickle(PATHS['tfidf'])
    svm_clf = load_pickle(PATHS['svm'])
    svm_train_time = load_timing('svm')
    X_tr = tfidf.transform(train_svm)
    X_vl = tfidf.transform(val_svm)
    X_te = tfidf.transform(test_svm)
else:
    print('Training TF-IDF + LinearSVC...')
    tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
    X_tr  = tfidf.fit_transform(train_svm)
    X_vl  = tfidf.transform(val_svm)
    X_te  = tfidf.transform(test_svm)
    t0 = time.time()
    svm_clf = CalibratedClassifierCV(LinearSVC(max_iter=5000, C=1.0, random_state=SEED), cv=3)
    svm_clf.fit(X_tr, train_labels)
    svm_train_time = time.time() - t0
    save_pickle(tfidf,   PATHS['tfidf'])
    save_pickle(svm_clf, PATHS['svm'])
    save_timing('svm', svm_train_time)
    status(f'SVM trained in {svm_train_time:.1f}s and saved', loaded=False)

val_f1_svm = f1_score(val_labels, svm_clf.predict(X_vl), average='macro', zero_division=0)
print(f'  Feature shape : {X_tr.shape}')
print(f'  Val F1-macro  : {val_f1_svm:.4f}')
print(f'  Train time    : {svm_train_time:.1f}s')

Training TF-IDF + LinearSVC...
  🏋️ TRAINED — SVM trained in 0.5s and saved
  Feature shape : (11000, 50000)
  Val F1-macro  : 0.8446
  Train time    : 0.5s


## Section 10 — Base Model Probability Predictions *(with checkpoint resume)*

Each model produces a `[P(neg), P(neu), P(pos)]` vector per sample. These are the meta-features for the stacking ensemble.

All six NumPy arrays (train / val / test × 3 models) are saved to `outputs/models/probas/` as `.npy` files and reloaded automatically on subsequent runs.

> This is the most time-consuming inference step — **loading from disk saves ~10–15 minutes** on subsequent runs.

In [15]:
# ── IndoBERT probabilities ────────────────────────────────────────────────────
if not FORCE_RETRAIN and probas_exist('ib'):
    status('IndoBERT probabilities found — loading .npy files', loaded=True)
    ib_train_proba = load_npy('ib', 'train')
    ib_val_proba   = load_npy('ib', 'val')
    ib_test_proba  = load_npy('ib', 'test')
    ib_true        = np.array(test_labels)
    ib_inf         = load_timing('ib_inf')
else:
    print('  [1/3] Running IndoBERT inference...')
    ib_train_proba, _, _            = get_probas_timed(indobert_model, indobert_tok, train_clean, train_labels)
    ib_val_proba,   _, _            = get_probas_timed(indobert_model, indobert_tok, val_clean,   val_labels)
    ib_test_proba,  ib_true, ib_inf = get_probas_timed(indobert_model, indobert_tok, test_clean,  test_labels)
    save_npy(ib_train_proba, 'ib', 'train'); save_npy(ib_val_proba, 'ib', 'val')
    save_npy(ib_test_proba,  'ib', 'test');  save_timing('ib_inf', ib_inf)
    status(f'IndoBERT probas saved (test inf={ib_inf:.2f}s)', loaded=False)

# ── IndoBERTweet probabilities ────────────────────────────────────────────────
if not FORCE_RETRAIN and probas_exist('ibt'):
    status('IndoBERTweet probabilities found — loading .npy files', loaded=True)
    ibt_train_proba = load_npy('ibt', 'train')
    ibt_val_proba   = load_npy('ibt', 'val')
    ibt_test_proba  = load_npy('ibt', 'test')
    ibt_true        = np.array(test_labels)
    ibt_inf         = load_timing('ibt_inf')
else:
    print('  [2/3] Running IndoBERTweet inference...')
    ibt_train_proba, _, _              = get_probas_timed(indobertweet_model, indobertweet_tok, train_clean, train_labels)
    ibt_val_proba,   _, _              = get_probas_timed(indobertweet_model, indobertweet_tok, val_clean,   val_labels)
    ibt_test_proba, ibt_true, ibt_inf  = get_probas_timed(indobertweet_model, indobertweet_tok, test_clean,  test_labels)
    save_npy(ibt_train_proba, 'ibt', 'train'); save_npy(ibt_val_proba, 'ibt', 'val')
    save_npy(ibt_test_proba,  'ibt', 'test');  save_timing('ibt_inf', ibt_inf)
    status(f'IndoBERTweet probas saved (test inf={ibt_inf:.2f}s)', loaded=False)

# ── SVM probabilities ─────────────────────────────────────────────────────────
if not FORCE_RETRAIN and probas_exist('svm'):
    status('SVM probabilities found — loading .npy files', loaded=True)
    svm_train_proba = load_npy('svm', 'train')
    svm_val_proba   = load_npy('svm', 'val')
    svm_test_proba  = load_npy('svm', 'test')
    svm_inf         = load_timing('svm_inf')
else:
    print('  [3/3] Running SVM inference...')
    t0 = time.time()
    svm_train_proba = svm_clf.predict_proba(X_tr)
    svm_val_proba   = svm_clf.predict_proba(X_vl)
    svm_test_proba  = svm_clf.predict_proba(X_te)
    svm_inf = time.time() - t0
    save_npy(svm_train_proba, 'svm', 'train'); save_npy(svm_val_proba, 'svm', 'val')
    save_npy(svm_test_proba,  'svm', 'test');  save_timing('svm_inf', svm_inf)
    status(f'SVM probas saved (test inf={svm_inf:.3f}s)', loaded=False)

print(f'\n✓ Shapes — IB:{ib_test_proba.shape}  IBT:{ibt_test_proba.shape}  SVM:{svm_test_proba.shape}')

  [1/3] Running IndoBERT inference...
  🏋️ TRAINED — IndoBERT probas saved (test inf=4.10s)
  [2/3] Running IndoBERTweet inference...
  🏋️ TRAINED — IndoBERTweet probas saved (test inf=4.07s)
  [3/3] Running SVM inference...
  🏋️ TRAINED — SVM probas saved (test inf=0.022s)

✓ Shapes — IB:(500, 3)  IBT:(500, 3)  SVM:(500, 3)


## Section 11 — Stacking Ensemble *(with checkpoint resume)*

The 9-dimensional meta-feature vector:

$$\mathbf{x}_{\text{meta}} = [P_{\text{IB}}^{\text{neg}}, P_{\text{IB}}^{\text{neu}}, P_{\text{IB}}^{\text{pos}},\; P_{\text{IBT}}^{\text{neg}}, P_{\text{IBT}}^{\text{neu}}, P_{\text{IBT}}^{\text{pos}},\; P_{\text{SVM}}^{\text{neg}}, P_{\text{SVM}}^{\text{neu}}, P_{\text{SVM}}^{\text{pos}}]$$

A **Logistic Regression** meta-learner is trained on training meta-features and evaluated on the test set. The trained meta-learner is saved as `meta_learner.pkl`.

> Logistic Regression is interpretable via SHAP — we can directly quantify each base model's contribution.

In [16]:
def stack(a, b, c): return np.hstack([a, b, c])

X_meta_tr = stack(ib_train_proba, ibt_train_proba, svm_train_proba)
X_meta_te = stack(ib_test_proba,  ibt_test_proba,  svm_test_proba)

if not FORCE_RETRAIN and ckpt_exists('meta'):
    status('Meta-learner checkpoint found — loading from disk', loaded=True)
    meta = load_pickle(PATHS['meta'])
else:
    print('Training Logistic Regression meta-learner...')
    meta = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, multi_class='multinomial')
    meta.fit(X_meta_tr, train_labels)
    save_pickle(meta, PATHS['meta'])
    status('Meta-learner trained and saved', loaded=False)

ens_preds = meta.predict(X_meta_te)
ens_proba = meta.predict_proba(X_meta_te)
test_true = np.array(test_labels)

print(f'\n  Meta-feature shape : train={X_meta_tr.shape}  test={X_meta_te.shape}')
print(f'  Ensemble F1-macro  : {f1_score(test_true, ens_preds, average="macro", zero_division=0):.4f}')

Training Logistic Regression meta-learner...
  🏋️ TRAINED — Meta-learner trained and saved

  Meta-feature shape : train=(11000, 9)  test=(500, 9)
  Ensemble F1-macro  : 0.8299


## Section 12 — Checkpoint Status Summary

After running Sections 7–11, all models and probability arrays are persisted to disk. This cell provides a **full inventory** of saved checkpoints so you can verify everything before proceeding to evaluation.

In [17]:
def file_size(path):
    if os.path.isfile(path):
        mb = os.path.getsize(path) / 1024**2
        return f'{mb:.1f} MB'
    elif os.path.isdir(path):
        total = sum(os.path.getsize(os.path.join(r, f))
                    for r, _, files in os.walk(path) for f in files)
        return f'{total/1024**2:.1f} MB'
    return 'MISSING'

print('='*55)
print('  CHECKPOINT INVENTORY')
print('='*55)
checks = [
    ('IndoBERT model dir',    PATHS['ib_dir']),
    ('IndoBERTweet model dir',PATHS['ibt_dir']),
    ('TF-IDF vectorizer',     PATHS['tfidf']),
    ('Calibrated SVM',        PATHS['svm']),
    ('Meta-learner (LR)',     PATHS['meta']),
    ('Timing JSON',           PATHS['timing']),
]
for name, path in checks:
    exists = os.path.isfile(path) or os.path.isdir(path)
    icon = '✓' if exists else '✗'
    print(f'  {icon}  {name:<28}  {file_size(path)}')

print()
proba_names = [f'ib_{s}' for s in ('train','val','test')] + \
              [f'ibt_{s}' for s in ('train','val','test')] + \
              [f'svm_{s}' for s in ('train','val','test')]
for pn in proba_names:
    path = f"{PATHS['probas_dir']}/{pn}.npy"
    exists = os.path.isfile(path)
    icon = '✓' if exists else '✗'
    print(f'  {icon}  probas/{pn}.npy  {file_size(path)}')

print('='*55)

  CHECKPOINT INVENTORY
  ✓  IndoBERT model dir            475.4 MB
  ✓  IndoBERTweet model dir        422.5 MB
  ✓  TF-IDF vectorizer             1.9 MB
  ✓  Calibrated SVM                3.4 MB
  ✓  Meta-learner (LR)             0.0 MB
  ✓  Timing JSON                   0.0 MB

  ✓  probas/ib_train.npy  0.1 MB
  ✓  probas/ib_val.npy  0.0 MB
  ✓  probas/ib_test.npy  0.0 MB
  ✓  probas/ibt_train.npy  0.1 MB
  ✓  probas/ibt_val.npy  0.0 MB
  ✓  probas/ibt_test.npy  0.0 MB
  ✓  probas/svm_train.npy  0.3 MB
  ✓  probas/svm_val.npy  0.0 MB
  ✓  probas/svm_test.npy  0.0 MB


## Section 13 — Model Evaluation

All four models evaluated on the held-out test set:

| Metric | Description |
|---|---|
| **Accuracy** | Correct / total |
| **F1-Macro** | Average F1 per class (unweighted) — primary metric |
| **F1-Micro** | Global TP/FP/FN pooled across classes |
| **Precision / Recall Macro** | Class-averaged precision and recall |

→ `model_metrics.csv` = **Table 3** · `test_predictions.csv` = full prediction dump

In [18]:
def full_eval(name, preds):
    r = {
        'model':           name,
        'accuracy':        round(accuracy_score(test_true, preds), 4),
        'f1_macro':        round(f1_score(test_true, preds, average='macro',  zero_division=0), 4),
        'f1_micro':        round(f1_score(test_true, preds, average='micro',  zero_division=0), 4),
        'precision_macro': round(precision_score(test_true, preds, average='macro', zero_division=0), 4),
        'recall_macro':    round(recall_score(test_true, preds, average='macro',    zero_division=0), 4),
    }
    print(f'\n  [{name}]')
    print(classification_report(test_true, preds, target_names=['negative','neutral','positive']))
    return r

ib_preds  = np.argmax(ib_test_proba,  1)
ibt_preds = np.argmax(ibt_test_proba, 1)
svm_preds = svm_clf.predict(X_te)

results = [
    full_eval('IndoBERT',     ib_preds),
    full_eval('IndoBERTweet', ibt_preds),
    full_eval('TF-IDF+SVM',   svm_preds),
    full_eval('Ensemble',     ens_preds),
]

metrics_df = pd.DataFrame(results)
metrics_df.to_csv('outputs/metrics/model_metrics.csv', index=False)
print('\n' + '='*55)
print(metrics_df.to_string(index=False))
print('\n✓ Saved model_metrics.csv')

pred_df = pd.DataFrame({
    'text':              test_texts,
    'true_label':        [ID2LABEL[l] for l in test_true],
    'indobert_pred':     [ID2LABEL[p] for p in ib_preds],
    'indobertweet_pred': [ID2LABEL[p] for p in ibt_preds],
    'svm_pred':          [ID2LABEL[p] for p in svm_preds],
    'ensemble_pred':     [ID2LABEL[p] for p in ens_preds],
})
pred_df.to_csv('outputs/predictions/test_predictions.csv', index=False)
print('✓ Saved test_predictions.csv')


  [IndoBERT]
              precision    recall  f1-score   support

    negative       0.94      0.94      0.94       208
     neutral       0.90      0.72      0.80        88
    positive       0.91      0.99      0.95       204

    accuracy                           0.92       500
   macro avg       0.92      0.88      0.89       500
weighted avg       0.92      0.92      0.92       500


  [IndoBERTweet]
              precision    recall  f1-score   support

    negative       0.95      0.96      0.95       208
     neutral       0.98      0.68      0.81        88
    positive       0.89      1.00      0.94       204

    accuracy                           0.93       500
   macro avg       0.94      0.88      0.90       500
weighted avg       0.93      0.93      0.92       500


  [TF-IDF+SVM]
              precision    recall  f1-score   support

    negative       0.89      0.83      0.86       208
     neutral       0.73      0.41      0.53        88
    positive       0.75    

### Figure 2 — Model Performance Comparison

Grouped bar chart comparing all five metrics across all four models.  
→ Insert in **Section 5 (Results)** of your paper.

In [19]:
met_list = ['accuracy','f1_macro','f1_micro','precision_macro','recall_macro']
x = np.arange(len(results)); w = 0.15
colors_b = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
fig, ax = plt.subplots(figsize=(14, 6))
for i, m in enumerate(met_list):
    ax.bar(x + i*w, [r[m] for r in results], w,
           label=m.replace('_',' ').title(), color=colors_b[i])
ax.set_xticks(x + w*2)
ax.set_xticklabels([r['model'] for r in results], fontsize=11)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/fig2_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig2_performance_comparison.png')

✓ Saved fig2_performance_comparison.png


### Figure 3 — Confusion Matrices

Normalised confusion matrices for all four models side-by-side. Cell values show **raw count** and **(normalised rate)**.  
→ Insert in **Section 5 (Results)** of your paper.

In [20]:
cls_n = ['negative','neutral','positive']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name, preds in zip(axes,
    ['IndoBERT','IndoBERTweet','TF-IDF+SVM','Ensemble'],
    [ib_preds, ibt_preds, svm_preds, ens_preds]):
    cm  = confusion_matrix(test_true, preds)
    cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im  = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(cls_n, rotation=45, ha='right')
    ax.set_yticklabels(cls_n)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(name, fontweight='bold')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i,j]}\n({cmn[i,j]:.2f})',
                    ha='center', va='center', fontsize=9,
                    color='white' if cmn[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig3_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig3_confusion_matrices.png')

✓ Saved fig3_confusion_matrices.png


## Section 14 — Architecture Diagram

This diagram illustrates the full stacking ensemble pipeline from raw text input to final prediction.

→ **Figure 4** — insert in **Section 3 (Methodology)** of your paper.

In [21]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')

def box(ax, x, y, w, h, label, color, fs=10):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                  linewidth=1.5, edgecolor='black', facecolor=color))
    ax.text(x+w/2, y+h/2, label, ha='center', va='center', fontsize=fs, fontweight='bold')

def arr(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))

box(ax, 5.5, 6.8, 3.0, 0.8, 'Raw Indonesian Social Media Text', '#AED6F1', fs=9)
box(ax, 0.2, 4.5, 3.2, 1.0, 'IndoBERT\n(indobert-base-p1)',      '#85C1E9')
box(ax, 5.0, 4.5, 3.2, 1.0, 'IndoBERTweet\n(indobertweet-base)', '#85C1E9')
box(ax, 9.8, 4.5, 3.2, 1.0, 'TF-IDF + LinearSVC\n(Calibrated)', '#A9DFBF')
arr(ax, 5.75, 6.8, 1.8,  5.5)
arr(ax, 7.00, 6.8, 6.6,  5.5)
arr(ax, 8.25, 6.8, 11.4, 5.5)
for xc in [1.8, 6.6, 11.4]:
    box(ax, xc-1.2, 3.2, 2.4, 0.8, 'P(neg, neu, pos)', '#FAD7A0', fs=8)
    arr(ax, xc, 4.5, xc, 4.0)
box(ax, 3.5, 2.0, 7.0, 0.8, 'Meta-feature vector  [9-dim: 3 × 3 class probabilities]', '#F9E79F')
for xc in [1.8, 6.6, 11.4]: arr(ax, xc, 3.2, 7.0, 2.8)
box(ax, 4.5, 0.8, 5.0, 0.9, 'Logistic Regression\n(Meta-Learner)', '#D7BDE2')
arr(ax, 7.0, 2.0, 7.0, 1.7)
box(ax, 5.5, 0.0, 3.0, 0.6, 'Final Prediction (3-class)', '#ABEBC6', fs=9)
arr(ax, 7.0, 0.8, 7.0, 0.6)

ax.set_title('Stacking Ensemble Architecture', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('outputs/figures/fig4_ensemble_architecture.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig4_ensemble_architecture.png')

✓ Saved fig4_ensemble_architecture.png


## Section 15 — Ablation Study

We remove one base model at a time to quantify each component's contribution.

| Experiment | Base Models Used |
|---|---|
| Full Ensemble | IndoBERT + IndoBERTweet + SVM |
| w/o SVM | IndoBERT + IndoBERTweet |
| w/o IndoBERTweet | IndoBERT + SVM |
| w/o IndoBERT | IndoBERTweet + SVM |

→ `ablation_results.csv` = **Table 5** · Figure 8 → **Section 5 (Results)**

In [22]:
def ablation_run(name, tr_feats, te_feats):
    m = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, multi_class='multinomial')
    m.fit(tr_feats, train_labels)
    p = m.predict(te_feats)
    return {
        'model':           name,
        'accuracy':        round(accuracy_score(test_true, p), 4),
        'f1_macro':        round(f1_score(test_true, p, average='macro',  zero_division=0), 4),
        'f1_micro':        round(f1_score(test_true, p, average='micro',  zero_division=0), 4),
        'precision_macro': round(precision_score(test_true, p, average='macro', zero_division=0), 4),
        'recall_macro':    round(recall_score(test_true, p, average='macro',    zero_division=0), 4),
    }

abl = [
    full_eval('Full Ensemble', ens_preds),
    ablation_run('w/o SVM',
                 np.hstack([ib_train_proba, ibt_train_proba]),
                 np.hstack([ib_test_proba,  ibt_test_proba])),
    ablation_run('w/o IndoBERTweet',
                 np.hstack([ib_train_proba,  svm_train_proba]),
                 np.hstack([ib_test_proba,   svm_test_proba])),
    ablation_run('w/o IndoBERT',
                 np.hstack([ibt_train_proba, svm_train_proba]),
                 np.hstack([ibt_test_proba,  svm_test_proba])),
]

abl_df = pd.DataFrame(abl)
abl_df.to_csv('outputs/metrics/ablation_results.csv', index=False)
print(abl_df.to_string(index=False))

# ── Figure 8 ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
met_abl = ['accuracy', 'f1_macro', 'f1_micro']
x_abl = np.arange(len(abl)); w_abl = 0.22
colors_abl = ['#2196F3', '#FF9800', '#4CAF50']
for i, m in enumerate(met_abl):
    ax.bar(x_abl + i*w_abl, [r[m] for r in abl], w_abl,
           label=m.replace('_',' ').title(), color=colors_abl[i], edgecolor='black')
ax.set_xticks(x_abl + w_abl)
ax.set_xticklabels([r['model'] for r in abl], fontsize=10)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Ablation Study: Component Contribution', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/fig8_ablation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved ablation_results.csv | fig8_ablation_comparison.png')


  [Full Ensemble]
              precision    recall  f1-score   support

    negative       0.94      0.92      0.93       208
     neutral       0.81      0.53      0.64        88
    positive       0.85      0.99      0.92       204

    accuracy                           0.88       500
   macro avg       0.87      0.82      0.83       500
weighted avg       0.88      0.88      0.87       500

           model  accuracy  f1_macro  f1_micro  precision_macro  recall_macro
   Full Ensemble     0.882    0.8299     0.882           0.8664        0.8158
         w/o SVM     0.920    0.8945     0.920           0.9160        0.8812
w/o IndoBERTweet     0.876    0.8243     0.876           0.8575        0.8109
    w/o IndoBERT     0.862    0.8120     0.862           0.8433        0.7997
✓ Saved ablation_results.csv | fig8_ablation_comparison.png


## Section 16 — SHAP Explainability

**SHAP** is applied to the Logistic Regression meta-learner (9-dim input → runs in seconds).

### Why the meta-learner, not the transformers?
- SHAP on full BERT models requires gradient-based perturbation over 110M parameters — slow
- SHAP on the 9-dim meta-learner directly answers: *"Which base model's class probability drove this prediction?"*
- `shap.LinearExplainer` is exact (not approximate) for linear models

### Feature labels
```
[IB_neg, IB_neu, IB_pos]   — IndoBERT outputs
[IBT_neg, IBT_neu, IBT_pos] — IndoBERTweet outputs
[SVM_neg, SVM_neu, SVM_pos] — SVM outputs
```

→ **Figure 5** + `shap_feature_importance.csv` → **Section 6 (Explainability)**

In [24]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

feat_names = [
    'IB_neg','IB_neu','IB_pos',
    'IBT_neg','IBT_neu','IBT_pos',
    'SVM_neg','SVM_neu','SVM_pos'
]

# Use smaller subset
X_shap = X_meta_te[:100]

# =========================
# SAFE SHAP EXPLAINER
# =========================
explainer = shap.Explainer(meta.predict_proba, X_meta_tr)

# Compute SHAP values
shap_values = explainer(X_shap)

print("SHAP values shape:", shap_values.values.shape)

# =========================
# PLOT
# =========================
class_names = ['Negative', 'Neutral', 'Positive']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ci, (ax, cn) in enumerate(zip(axes, class_names)):

    sv = shap_values.values[:, :, ci]

    mean_abs = np.abs(sv).mean(axis=0)

    sort_idx = np.argsort(mean_abs)[::-1]

    ax.barh(
        range(len(feat_names)),
        mean_abs[sort_idx],
        color='#1565C0',
        edgecolor='black'
    )

    ax.set_yticks(range(len(feat_names)))
    ax.set_yticklabels(
        [feat_names[i] for i in sort_idx],
        fontsize=9
    )

    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'{cn} Class', fontweight='bold')

    ax.grid(axis='x', alpha=0.3)

plt.suptitle(
    'SHAP Feature Importance — Meta-Learner (Logistic Regression)',
    fontsize=13,
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    'outputs/figures/fig5_shap_summary.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# =========================
# SAVE CSV
# =========================
shap_df = pd.DataFrame(
    {
        cn.lower(): np.abs(shap_values.values[:, :, i]).mean(axis=0)
        for i, cn in enumerate(class_names)
    },
    index=feat_names
)

shap_df.to_csv(
    'outputs/metrics/shap_feature_importance.csv'
)

print('✓ Saved fig5_shap_summary.png')
print('✓ Saved shap_feature_importance.csv')

ExactExplainer explainer: 101it [00:16,  5.97it/s]                        


SHAP values shape: (100, 9, 3)
✓ Saved fig5_shap_summary.png
✓ Saved shap_feature_importance.csv


## Section 17 — Token Importance Visualization

We compute **gradient × embedding** saliency for IndoBERT:

$$\text{importance}(t_i) = \left\| \frac{\partial \hat{y}_{\text{true}}}{\partial \mathbf{e}_i} \odot \mathbf{e}_i \right\|_1$$

### Why gradient × embedding over raw attention?
- Attention weights show *where* the model looks, not *what drives the decision*
- Gradient-based saliency is more faithful to the model computation
- Compatible with any BERT-family architecture without modification

One correctly-predicted example is shown per class (3 rows).

→ **Figure 6** → **Section 6 (Explainability)**

In [25]:
def token_importance(model, tok, text, lbl, max_len=MAX_LEN):
    model.eval()
    enc  = tok(text, return_tensors='pt', max_length=max_len,
               padding='max_length', truncation=True)
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    emb  = model.base_model.embeddings.word_embeddings(ids).detach().requires_grad_(True)
    out  = model(inputs_embeds=emb, attention_mask=mask)
    out.logits[0, lbl].backward()
    scores = (emb.grad[0].detach().cpu().numpy() * emb[0].detach().cpu().numpy()).sum(-1)
    scores = np.abs(scores); scores /= (scores.max() + 1e-9)
    tokens = tok.convert_ids_to_tokens(ids[0].cpu())
    seq_len = mask[0].sum().item()
    return tokens[:seq_len], scores[:seq_len]

examples = []
for tgt in [0, 1, 2]:
    for i, (txt, lbl) in enumerate(zip(test_clean, test_labels)):
        if lbl == tgt and ens_preds[i] == tgt:
            examples.append((txt, lbl)); break

fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, (txt, lbl) in zip(axes, examples):
    try:
        tokens, scores = token_importance(indobert_model, indobert_tok, txt, lbl)
        tokens = tokens[:40]; scores = scores[:40]; n = len(tokens)
        ax.imshow([scores], cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
        ax.set_xticks(range(n))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
        ax.set_yticks([])
        ax.set_title(f"True: {ID2LABEL[lbl]}  |  {txt[:70]}...", fontsize=9)
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {e}', ha='center', transform=ax.transAxes)
plt.suptitle('Token Importance (Gradient × Embedding) — IndoBERT',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig6_token_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig6_token_importance.png')

✓ Saved fig6_token_importance.png


## Section 18 — Efficiency Analysis

| Metric | Description |
|---|---|
| **Parameters (M)** | Total trainable parameters |
| **Train time (s)** | Wall-clock fine-tuning time (restored from `timing.json`) |
| **Inference time (s)** | Total time to score the test set |
| **Latency (ms/sample)** | Average per-sample inference latency |
| **GPU Memory (MB)** | Peak GPU memory during inference |

> Training times are loaded from `timing.json` — so this section works correctly even on the second run when training was skipped.

→ `efficiency_analysis.csv` = **Table 6** · Figure 7 → **Section 5 (Discussion)**

In [26]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
_, _, ib_inf2  = get_probas_timed(indobert_model,     indobert_tok,     test_clean, test_labels)
ib_gpu = gpu_mb()

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
_, _, ibt_inf2 = get_probas_timed(indobertweet_model, indobertweet_tok, test_clean, test_labels)
ibt_gpu = gpu_mb()

t0 = time.time(); _ = svm_clf.predict_proba(X_te); svm_inf2 = time.time() - t0

n   = len(test_texts)
lat = lambda t: round(t / n * 1000, 3)

# Restore training times from timing.json (works on resume runs)
ib_train_time_r  = load_timing('indobert',    default=ib_train_time)
ibt_train_time_r = load_timing('indobertweet',default=ibt_train_time)
svm_train_time_r = load_timing('svm',         default=svm_train_time)

eff_data = [
    {'model':'IndoBERT',
     'params_M': round(count_params(indobert_model)/1e6, 1),
     'train_s':  round(ib_train_time_r, 1),
     'inf_s':    round(ib_inf2, 3),
     'lat_ms':   lat(ib_inf2),
     'gpu_MB':   round(ib_gpu, 1),
     'f1_macro': metrics_df[metrics_df.model=='IndoBERT']['f1_macro'].values[0]},
    {'model':'IndoBERTweet',
     'params_M': round(count_params(indobertweet_model)/1e6, 1),
     'train_s':  round(ibt_train_time_r, 1),
     'inf_s':    round(ibt_inf2, 3),
     'lat_ms':   lat(ibt_inf2),
     'gpu_MB':   round(ibt_gpu, 1),
     'f1_macro': metrics_df[metrics_df.model=='IndoBERTweet']['f1_macro'].values[0]},
    {'model':'TF-IDF+SVM',
     'params_M': 0.05,
     'train_s':  round(svm_train_time_r, 1),
     'inf_s':    round(svm_inf2, 3),
     'lat_ms':   lat(svm_inf2),
     'gpu_MB':   0.0,
     'f1_macro': metrics_df[metrics_df.model=='TF-IDF+SVM']['f1_macro'].values[0]},
    {'model':'Ensemble',
     'params_M': round((count_params(indobert_model)+count_params(indobertweet_model))/1e6+0.05, 1),
     'train_s':  round(ib_train_time_r+ibt_train_time_r+svm_train_time_r, 1),
     'inf_s':    round(ib_inf2+ibt_inf2+svm_inf2, 3),
     'lat_ms':   lat(ib_inf2+ibt_inf2+svm_inf2),
     'gpu_MB':   round(ib_gpu+ibt_gpu, 1),
     'f1_macro': metrics_df[metrics_df.model=='Ensemble']['f1_macro'].values[0]},
]

eff_df = pd.DataFrame(eff_data)
eff_df.to_csv('outputs/metrics/efficiency_analysis.csv', index=False)
print(eff_df.to_string(index=False))

# ── Figure 7 ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
scatter_colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
for row, c in zip(eff_data, scatter_colors):
    ax.scatter(row['lat_ms'], row['f1_macro'], s=220, color=c, zorder=5, edgecolors='black')
    ax.annotate(row['model'], (row['lat_ms'], row['f1_macro']),
                textcoords='offset points', xytext=(8, 4), fontsize=10)
ax.set_xlabel('Latency (ms / sample)', fontsize=12)
ax.set_ylabel('F1 Macro', fontsize=12)
ax.set_title('Efficiency vs. Performance Trade-off', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/fig7_efficiency_vs_performance.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved efficiency_analysis.csv | fig7_efficiency_vs_performance.png')

       model  params_M  train_s  inf_s  lat_ms  gpu_MB  f1_macro
    IndoBERT    124.40    833.8  4.021   8.043  1311.7    0.8944
IndoBERTweet    110.60    823.3  4.103   8.206  1311.7    0.9015
  TF-IDF+SVM      0.05      0.5  0.004   0.008     0.0    0.7410
    Ensemble    235.10   1657.6  8.128  16.256  2623.5    0.8299
✓ Saved efficiency_analysis.csv | fig7_efficiency_vs_performance.png


## Section 19 — Failure Case Analysis

We categorise ensemble misclassifications by failure pattern:

| Type | Signal | Examples |
|---|---|---|
| **Sarcasm** | `wkwk`, `haha`, `padahal`, 😂 | Ironic positives |
| **Code-mixed** | English function words | `the`, `is`, `we` |
| **Slang** | ≥ 2 slang dict hits | `bgt`, `gak`, `yg` |
| **Other** | None of the above | Ambiguous phrasing |

→ `failure_cases.csv` — supports the **Error Analysis** discussion in your paper.

In [27]:
miss = np.where(ens_preds != test_true)[0]

def tag_failure(text):
    t = text.lower()
    tags = []
    if re.search(r'(wkwk|haha|lol|\U0001f602|\U0001f605|ya kan|padahal)', t):
        tags.append('sarcasm')
    if re.search(r'\b(the|is|are|i |my |you |we )\b', t):
        tags.append('code-mixed')
    if sum(1 for w in t.split() if w in SLANG_DICT) >= 2:
        tags.append('slang')
    return '|'.join(tags) if tags else 'other'

fail_df = pd.DataFrame({
    'text':      [test_texts[i] for i in miss],
    'true':      [ID2LABEL[test_true[i]]  for i in miss],
    'predicted': [ID2LABEL[ens_preds[i]]  for i in miss],
    'type':      [tag_failure(test_texts[i]) for i in miss],
})
fail_df.to_csv('outputs/predictions/failure_cases.csv', index=False)

print(f'Total misclassified : {len(fail_df)} / {len(test_true)} ({len(fail_df)/len(test_true)*100:.1f}%)')
print('\nFailure type breakdown:')
print(fail_df['type'].value_counts().to_string())
print('\nSample failure cases:')
for _, row in fail_df.head(8).iterrows():
    print(f"  [{row['type']:12s}] True={row['true']:8s} Pred={row['predicted']:8s} | {row['text'][:65]}")

Total misclassified : 59 / 500 (11.8%)

Failure type breakdown:
type
other            57
sarcasm|slang     1
sarcasm           1

Sample failure cases:
  [other       ] True=positive Pred=neutral  | takdir politik ahy belum bisa ikut kontestasi pilpres 2019 - 2024
  [sarcasm|slang] True=negative Pred=positive | temen gue telepon genggam nya xiaomi yang 2 juta saja masih kemba
  [other       ] True=negative Pred=neutral  | terima kasih kepada anies baswedan dan diego uno atas komitmen ny
  [sarcasm     ] True=negative Pred=positive | iya enak banget soal nya ; iya dong indomie mah teh terbaik . iya
  [other       ] True=negative Pred=neutral  | kalau driver go jek tidak pelayanan prima meminta bungkus acar un
  [other       ] True=negative Pred=neutral  | terima kasih atas saran yang kamu berikan kak . kami akan selalu 
  [other       ] True=negative Pred=neutral  | hai kak , terima kasih atas masukan kamu ya . saran kamu sangat b
  [other       ] True=negative Pred=positive | harus bel

## Section 20 — Per-Class Metrics Table

Detailed per-class precision, recall, and F1 for every model.

→ `per_class_metrics.csv` = **Table 4** in your paper.

In [28]:
rows = []
for nm, preds in [('IndoBERT',ib_preds),('IndoBERTweet',ibt_preds),
                  ('TF-IDF+SVM',svm_preds),('Ensemble',ens_preds)]:
    for ci, cn in enumerate(['negative','neutral','positive']):
        p = precision_score(test_true, preds, labels=[ci], average=None, zero_division=0)[0]
        r = recall_score(test_true,    preds, labels=[ci], average=None, zero_division=0)[0]
        f = f1_score(test_true,        preds, labels=[ci], average=None, zero_division=0)[0]
        rows.append({'model':nm,'class':cn,
                     'precision':round(p,4),'recall':round(r,4),'f1':round(f,4)})
pc_df = pd.DataFrame(rows)
pc_df.to_csv('outputs/metrics/per_class_metrics.csv', index=False)
print(pc_df.to_string(index=False))
print('\n✓ Saved per_class_metrics.csv')

       model    class  precision  recall     f1
    IndoBERT negative     0.9375  0.9375 0.9375
    IndoBERT  neutral     0.9000  0.7159 0.7975
    IndoBERT positive     0.9099  0.9902 0.9484
IndoBERTweet negative     0.9479  0.9615 0.9547
IndoBERTweet  neutral     0.9836  0.6818 0.8054
IndoBERTweet positive     0.8947  1.0000 0.9444
  TF-IDF+SVM negative     0.8866  0.8269 0.8557
  TF-IDF+SVM  neutral     0.7347  0.4091 0.5255
  TF-IDF+SVM positive     0.7549  0.9510 0.8416
    Ensemble negative     0.9366  0.9231 0.9298
    Ensemble  neutral     0.8103  0.5341 0.6438
    Ensemble positive     0.8523  0.9902 0.9161

✓ Saved per_class_metrics.csv


## Final Summary — All Outputs

---

### Paper → Output Mapping

#### Tables

| Paper Table | Description | File |
|---|---|---|
| Table 2 | Dataset Statistics | `outputs/metrics/dataset_stats.csv` |
| Table 3 | Model Comparison | `outputs/metrics/model_metrics.csv` |
| Table 4 | Per-class P/R/F1 | `outputs/metrics/per_class_metrics.csv` |
| Table 5 | Ablation Study | `outputs/metrics/ablation_results.csv` |
| Table 6 | Efficiency | `outputs/metrics/efficiency_analysis.csv` |
| — | SHAP Importance | `outputs/metrics/shap_feature_importance.csv` |

#### Figures (all 300 DPI PNG)

| Fig | Description | File | Paper Section |
|---|---|---|---|
| 1 | Class Distribution | `fig1_class_distribution.png` | Dataset |
| 2 | Performance Bar Chart | `fig2_performance_comparison.png` | Results |
| 3 | Confusion Matrices | `fig3_confusion_matrices.png` | Results |
| 4 | Architecture Diagram | `fig4_ensemble_architecture.png` | Methodology |
| 5 | SHAP Summary | `fig5_shap_summary.png` | Explainability |
| 6 | Token Importance | `fig6_token_importance.png` | Explainability |
| 7 | Efficiency Scatter | `fig7_efficiency_vs_performance.png` | Discussion |
| 8 | Ablation Chart | `fig8_ablation_comparison.png` | Ablation |

#### Predictions
- `outputs/predictions/test_predictions.csv` — all test predictions per model
- `outputs/predictions/failure_cases.csv` — misclassified samples with failure type tags

#### Checkpoints (persisted across sessions if using Google Drive)
- `outputs/models/indobert/` — fine-tuned IndoBERT
- `outputs/models/indobertweet/` — fine-tuned IndoBERTweet
- `outputs/models/tfidf.pkl` — TF-IDF vectorizer
- `outputs/models/svm_clf.pkl` — Calibrated SVM
- `outputs/models/meta_learner.pkl` — LR meta-learner
- `outputs/models/timing.json` — training wall-clock times
- `outputs/models/probas/*.npy` — base model probability arrays

---

### Reproducibility Checklist
- [x] Random seed **42** fixed across all libraries
- [x] Official IndoNLU SmSA train / val / test splits
- [x] Best checkpoint saved per **validation F1-macro**
- [x] All metrics from **real model outputs** — no fabricated numbers
- [x] All figures at **300 DPI** — publication-ready
- [x] All models and arrays **persist to disk** — safe to interrupt and resume

In [29]:
print('='*60)
print('  ALL EXPERIMENTS COMPLETE')
print('='*60)

for folder in ['outputs/metrics', 'outputs/figures', 'outputs/predictions']:
    print(f'\n{folder}/')
    for fn in sorted(os.listdir(folder)):
        fpath = os.path.join(folder, fn)
        size  = os.path.getsize(fpath) / 1024
        print(f'  {fn:<48}  {size:6.1f} KB')

print('\noutputs/models/ (checkpoints)')
for fn in sorted(os.listdir('outputs/models')):
    fpath = os.path.join('outputs/models', fn)
    if os.path.isdir(fpath):
        total = sum(os.path.getsize(os.path.join(r,f))
                    for r,_,files in os.walk(fpath) for f in files)
        print(f'  {fn}/ (dir)  {total/1024**2:.1f} MB')
    else:
        print(f'  {fn}  {os.path.getsize(fpath)/1024:.1f} KB')

print('\n✓ Run complete. Set FORCE_RETRAIN=False to resume from checkpoints next time.')

  ALL EXPERIMENTS COMPLETE

outputs/metrics/
  ablation_results.csv                                 0.2 KB
  dataset_stats.csv                                    0.1 KB
  efficiency_analysis.csv                              0.2 KB
  model_metrics.csv                                    0.2 KB
  per_class_metrics.csv                                0.5 KB
  shap_feature_importance.csv                          0.6 KB

outputs/figures/
  fig1_class_distribution.png                        154.5 KB
  fig2_performance_comparison.png                    110.9 KB
  fig3_confusion_matrices.png                        262.7 KB
  fig4_ensemble_architecture.png                     244.7 KB
  fig5_shap_summary.png                              170.8 KB
  fig6_token_importance.png                          446.6 KB
  fig7_efficiency_vs_performance.png                 122.1 KB
  fig8_ablation_comparison.png                        90.1 KB

outputs/predictions/
  failure_cases.csv                            